# Práctica 2 — Multiagente con LangGraph

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase04/notebooks/multiagente_langgraph.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Objetivo:** construir un mini-equipo de agentes con el patrón **orquestador** — un
> *manager* que recibe una tarea, la delega en especialistas (Investigador, Redactor,
> Revisor) y junta los resultados en una respuesta final.
>
> Esta es la **Práctica 2** de la Clase 4 y es la secuela de la **Práctica 1**
> (`react_tools_scratch.ipynb`), donde construimos un agente ReAct desde cero. Acá
> pasamos de **un** agente a **varios** agentes coordinados con **LangGraph**.
>
> El notebook es **autocontenido**: redefinimos las tools que necesitemos acá adentro,
> no importamos nada de la Práctica 1. Corre en Colab con una `GROQ_API_KEY` (u Ollama local).
>
> **Tiempo:** ~25 min de hands-on.

## 0. Setup

Instalá dependencias y configurá la API key de Groq.

In [ ]:
!pip install -q groq tavily-python python-dotenv langgraph langchain langchain-groq grandalf

In [ ]:
# OPCIONAL — sólo para correr en local. En Colab esta celda se autodetecta y se omite
# (en Colab configurá GROQ_API_KEY desde "Secrets" en la barra lateral izquierda).
#
# Si vos corrés esta notebook localmente con tu venv:
#   1. Poné GROQ_API_KEY=tu-api-key en un archivo .env en la raíz del repo
#   2. Esta celda lo carga automáticamente

try:
    import google.colab  # noqa: F401
    print('ℹ️  Colab detectado — saltando .env (usá Colab Secrets para GROQ_API_KEY).')
except ImportError:
    try:
        from dotenv import load_dotenv, find_dotenv
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
        from dotenv import load_dotenv, find_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f'✓ .env cargado desde {env_path}')
    else:
        print('ℹ️  No se encontró .env. Asegurate de exportar GROQ_API_KEY como env var antes de lanzar jupyter.')

In [ ]:
import os

try:
    from google.colab import userdata
    if 'GROQ_API_KEY' not in os.environ:
        try:
            os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
        except Exception:
            pass
    # Tavily es opcional — si la tenés en Secrets, la levantamos también.
    if 'TAVILY_API_KEY' not in os.environ:
        try:
            os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
        except Exception:
            pass
except ImportError:
    pass

assert os.environ.get('GROQ_API_KEY'), (
    'Falta GROQ_API_KEY. Configurala en Colab Secrets o como env var.'
)
print('✓ GROQ_API_KEY configurada')
print('✓ TAVILY_API_KEY' if os.environ.get('TAVILY_API_KEY') else 'ℹ️  Sin TAVILY_API_KEY — el Investigador usará un MOCK (corre igual)')

## El patrón orquestador

Un solo agente con muchas herramientas y un prompt gigante se vuelve **frágil**: pierde
foco y satura su ventana de contexto. El patrón **orquestador** divide para conquistar:
un agente *manager* recibe la tarea, la **delega** en especialistas y **junta** lo que
producen.

```
                       ┌──────────────────┐
            tarea  ──►  │   ORQUESTADOR    │   prepara la tarea y la pasa
                       └────────┬─────────┘
                                │
              ┌─────────────────┼─────────────────┐
              ▼                 ▼                 ▼
      ┌──────────────┐  ┌──────────────┐  ┌──────────────┐
      │ INVESTIGADOR │─►│   REDACTOR   │─►│   REVISOR    │─► resultado
      │ (busca info) │  │ (escribe el  │  │ (critica y   │     final
      │ buscar_web() │  │   borrador)  │  │   corrige)   │
      └──────────────┘  └──────────────┘  └──────────────┘
```

**¿Por qué varios agentes en vez de uno?**

- **Especialización** — cada agente tiene un rol acotado, pocas tools y un prompt afilado
  (el Investigador busca, el Redactor escribe, el Revisor corrige). Cada uno hace una cosa bien.
- **Contexto separado** — cada agente razona con su propio prompt; no compiten por espacio
  en una sola ventana de contexto gigante.
- **Modularidad** — podés testear, reemplazar o reusar cada agente por separado. Cambiar al
  Redactor no toca al Investigador.

**El tradeoff (multiagente no es gratis):** cada agente es **otra llamada al LLM**, así que
sumás **latencia** y **tokens**, más el costo de **coordinarlos**. Se justifica cuando la
tarea es genuinamente heterogénea (investigar + redactar + revisar son trabajos distintos).
La regla: empezá con un agente; pasá a multiagente cuando el agente único se rompe.

## 1. Herramientas (inline, autocontenidas)

El Investigador necesita **buscar en la web**. Definimos `buscar_web` acá mismo: usa
**Tavily** si hay `TAVILY_API_KEY`, y si no, cae a un **MOCK** etiquetado para que el
notebook corra igual sin credenciales externas.

La envolvemos con `@tool` de `langchain_core.tools` para poder dársela a un agente como
herramienta estándar de LangChain (con nombre + descripción + schema de argumentos).

In [ ]:
from langchain_core.tools import tool


@tool
def buscar_web(query: str) -> str:
    """Busca información en la web sobre 'query' y devuelve un resumen de resultados.

    Usá esta herramienta cuando necesites datos, definiciones o ejemplos actualizados
    que no estén en tu conocimiento interno.
    """
    api_key = os.environ.get('TAVILY_API_KEY')
    if api_key:
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=api_key)
            resp = client.search(query=query, max_results=3, include_answer=True)
            partes = []
            if resp.get('answer'):
                partes.append(f"Resumen: {resp['answer']}")
            for r in resp.get('results', []):
                partes.append(f"- {r['title']}: {r['content'][:300]}")
            return '\n'.join(partes) if partes else 'Sin resultados.'
        except Exception as e:
            return f'[ERROR Tavily: {e}] — seguí con tu conocimiento interno.'

    # ── MOCK (sin TAVILY_API_KEY): resultados filtrados pero útiles para la demo ──
    return (
        "[RESULTADO MOCK — sin TAVILY_API_KEY, búsqueda simulada]\n"
        f"Consulta: {query}\n"
        "- Un 'agente LLM' es un sistema que usa un modelo de lenguaje como motor de "
        "razonamiento dentro de un loop: percibe, decide qué hacer, ejecuta herramientas "
        "(APIs, búsqueda, código) y observa el resultado, repitiendo hasta cumplir un objetivo.\n"
        "- A diferencia de un chatbot (prompt -> texto, una sola vez), el agente PUEDE actuar "
        "sobre el mundo y encadenar pasos por su cuenta.\n"
        "- Sirve para tareas como: investigación con búsqueda web, asistentes de programación, "
        "automatización de flujos, y RAG agéntico (recuperar y razonar sobre documentos)."
    )


# Probamos la tool (las @tool se invocan con .invoke({...}) en LangChain)
print(buscar_web.invoke({'query': '¿qué es un agente LLM?'})[:400], '...')

## 2. El modelo (LLM)

Usamos **Groq** vía `langchain_groq` por su latencia muy baja (ideal para multiagente,
donde encadenamos varias llamadas). El modelo `llama-3.3-70b-versatile` es lo bastante
capaz para razonar y redactar.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.3-70b-versatile',   # modelo capaz, baja latencia en Groq
    temperature=0.3,                   # algo de creatividad, pero estable
)

# Alternativas:
#   - Groq más rápido/barato:  model='llama-3.1-8b-instant'
#   - Ollama local (sin API key, privado):
#         # pip install langchain-ollama  ;  ollama pull qwen3:8b
#         from langchain_ollama import ChatOllama
#         llm = ChatOllama(model='qwen3:8b', temperature=0.3)

print(llm.invoke('Decí "ok" si me podés escuchar.').content)

## 3. El estado compartido

En LangGraph todos los nodos leen y escriben sobre un **estado compartido**. Lo definimos
con un `TypedDict`: cada nodo recibe el estado actual y devuelve un **dict parcial** con
los campos que actualiza (LangGraph hace el *merge*).

Nuestro estado lleva la tarea original y lo que va produciendo cada especialista, para que
al final podamos **ver la contribución de cada agente**.

In [ ]:
from typing import TypedDict


class EstadoEquipo(TypedDict):
    tarea: str           # la consigna original (la setea el orquestador)
    investigacion: str   # lo que junta el Investigador
    borrador: str        # lo que escribe el Redactor
    revision: str        # las críticas/correcciones del Revisor
    final: str           # la versión final, ya corregida


print('✓ Estado compartido definido: tarea -> investigacion -> borrador -> revision -> final')

## 4. Los nodos: orquestador + 3 especialistas

Cada **nodo es una función** `f(state) -> dict_parcial`. Modelamos un flujo **orquestado
lineal** (es el patrón orquestador en su forma más simple y clara):

```
orquestador → investigador → redactor → revisor → END
```

El **orquestador** prepara la tarea (acá, en su forma mínima, la normaliza y la registra;
en sistemas más complejos también decidiría *qué* especialistas usar y en qué orden). Cada
especialista tiene un **system prompt afilado** para su rol y trabaja sobre lo que dejó el
anterior en el estado.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage


def _llm(system: str, user: str) -> str:
    """Helper: una llamada al LLM con system + user prompt, devuelve texto."""
    resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return resp.content


# ── ORQUESTADOR ── prepara/normaliza la tarea y la deja lista para el equipo.
def orquestador(state: EstadoEquipo) -> dict:
    tarea = state['tarea'].strip()
    print('🧭 [ORQUESTADOR] Recibí la tarea y la delego al equipo:')
    print(f'    "{tarea}"\n')
    return {'tarea': tarea}


# ── INVESTIGADOR ── usa buscar_web() para juntar info sobre la tarea.
def investigador(state: EstadoEquipo) -> dict:
    print('🔎 [INVESTIGADOR] Buscando información...')
    hallazgos = buscar_web.invoke({'query': state['tarea']})
    # El LLM sintetiza los resultados crudos en notas ordenadas.
    notas = _llm(
        system=('Sos un investigador. Te paso resultados de búsqueda web crudos. '
                'Sintetizá los puntos clave en 4-6 bullets claros y verificables, en español. '
                'No inventes: usá sólo lo que está en los resultados.'),
        user=f'Tarea: {state["tarea"]}\n\nResultados de búsqueda:\n{hallazgos}',
    )
    print('   ✓ Investigación lista.\n')
    return {'investigacion': notas}


# ── REDACTOR ── escribe un borrador a partir de la investigación.
def redactor(state: EstadoEquipo) -> dict:
    print('✍️  [REDACTOR] Escribiendo el borrador...')
    borrador = _llm(
        system=('Sos un redactor técnico. Escribí un texto claro y bien estructurado en '
                'español que cumpla la tarea pedida, basándote en las notas de investigación. '
                'Sé conciso y concreto.'),
        user=f'Tarea: {state["tarea"]}\n\nNotas de investigación:\n{state["investigacion"]}',
    )
    print('   ✓ Borrador listo.\n')
    return {'borrador': borrador}


# ── REVISOR ── critica el borrador y devuelve una versión final corregida.
def revisor(state: EstadoEquipo) -> dict:
    print('🔍 [REVISOR] Revisando y corrigiendo...')
    critica = _llm(
        system=('Sos un revisor exigente. Leé el borrador y señalá en 2-4 bullets qué se '
                'puede mejorar (claridad, precisión, faltantes). En español, breve.'),
        user=f'Tarea: {state["tarea"]}\n\nBorrador:\n{state["borrador"]}',
    )
    final = _llm(
        system=('Sos un editor. Reescribí el borrador aplicando las críticas para producir '
                'la VERSIÓN FINAL: clara, correcta y completa. Devolvé sólo el texto final, '
                'en español, sin meta-comentarios.'),
        user=(f'Tarea: {state["tarea"]}\n\nBorrador:\n{state["borrador"]}'
              f'\n\nCríticas a aplicar:\n{critica}'),
    )
    print('   ✓ Revisión y versión final listas.\n')
    return {'revision': critica, 'final': final}


print('✓ 4 nodos definidos: orquestador, investigador, redactor, revisor')

## 5. Armamos el grafo con LangGraph

Creamos el `StateGraph` con nuestro estado, agregamos los nodos, conectamos las aristas
desde `START` y hasta `END`, y compilamos. La API actual de LangGraph:

- `StateGraph(EstadoEquipo)` — el grafo tipado con el estado compartido.
- `add_node('nombre', funcion)` — registra cada nodo.
- `add_edge(desde, hasta)` — arista dirigida; `START` y `END` son los extremos.
- `.compile()` — devuelve un grafo ejecutable (`invoke`, `stream`, ...).

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(EstadoEquipo)

# 1) nodos
builder.add_node('orquestador', orquestador)
builder.add_node('investigador', investigador)
builder.add_node('redactor', redactor)
builder.add_node('revisor', revisor)

# 2) aristas: flujo orquestado lineal
builder.add_edge(START, 'orquestador')
builder.add_edge('orquestador', 'investigador')
builder.add_edge('investigador', 'redactor')
builder.add_edge('redactor', 'revisor')
builder.add_edge('revisor', END)

# 3) compilar
app = builder.compile()
print('✓ Grafo compilado')

# (opcional) ver la estructura del grafo: PNG en Colab; ascii o texto como fallback
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    try:
        print(app.get_graph().draw_ascii())   # requiere: pip install grandalf
    except Exception:
        print('Flujo: START -> orquestador -> investigador -> redactor -> revisor -> END')

## 6. Lo corremos

Invocamos el grafo con una tarea. Vamos a ver, paso a paso, lo que imprime cada agente
mientras trabaja, y al final mostramos **la contribución de cada uno** sobre el estado
compartido.

In [ ]:
resultado = app.invoke({
    'tarea': 'Escribí un resumen corto sobre qué es un agente LLM y para qué sirve, con un ejemplo.'
})

sep = '═' * 78


def bloque(titulo, texto):
    print(sep)
    print(f'  {titulo}')
    print(sep)
    print(texto)
    print()


bloque('① INVESTIGACIÓN (Investigador)', resultado['investigacion'])
bloque('② BORRADOR (Redactor)', resultado['borrador'])
bloque('③ REVISIÓN — críticas (Revisor)', resultado['revision'])
bloque('④ VERSIÓN FINAL (Revisor)', resultado['final'])

## 7. Reflexión — orquestador vs un solo agente

Acabás de ver cómo un *manager* delega en tres especialistas y arma la respuesta juntando
sus aportes. Cada agente tuvo un **rol acotado**, su **propio prompt** y su **propio
contexto** — y el resultado final pasó por una etapa explícita de **revisión**.

**¿Qué ganamos respecto de un solo agente?**

- Cada paso es **trazable**: ves la investigación, el borrador y la corrección por separado.
- Cada agente es **reemplazable**: podés mejorar al Redactor sin tocar al Investigador.
- El **contexto** de cada uno queda acotado a su tarea, no a un prompt monolítico.

**¿Qué nos costó?** Cuatro llamadas al LLM en lugar de una: más **latencia** y más **tokens**.
Para una tarea simple como esta, un único agente bien prompteado probablemente alcanzaría.

### La regla práctica

> **Empezá con un agente.** Pasá a multiagente **cuando el agente único se rompe** por
> complejidad (demasiadas herramientas, roles que se pisan) o por contexto (el prompt no
> entra, o se confunde entre subtareas). **No al revés:** multiagente agrega coordinación,
> latencia y costo — solo conviene cuando la tarea es genuinamente heterogénea.

**Para seguir explorando:**

- Cambiá el flujo lineal por un **orquestador que decide el ruteo** (con `add_conditional_edges`):
  por ejemplo, que el Revisor pueda **mandar el borrador de vuelta** al Redactor si no pasa.
- Sumale más especialistas (un Verificador de datos, un Traductor) y observá cómo escala.
- Dale herramientas distintas a cada agente (el RAG de la Clase 3 como tool del Investigador).

## 🚀 Mejora propuesta: un orquestador *de verdad* (supervisor)

Lo que armamos arriba es, en rigor, una **secuencia fija** (`orquestador → investigador → redactor → revisor → END`): el orquestador solo normaliza la tarea y el orden está **cableado a mano**. Eso es un **workflow**, no un agente que decide.

Un **orquestador real** (patrón *supervisor*) es **hub-and-spoke**: el control **vuelve a él después de cada agente** y él **decide dinámicamente** a quién llamar — y **cuándo terminar**. Eso habilita **loops**: si el revisor pide cambios, puede volver a redactar o a investigar.

| Secuencia (lo de arriba) | Supervisor (esta mejora) |
|---|---|
| aristas fijas A→B→C | cada worker vuelve al orquestador |
| el orden lo cableás vos | el orquestador **decide** el próximo paso |
| sin loops | puede **re-delegar** y reintentar |

Es la distinción de Anthropic en *Building Effective Agents*: **workflow** (caminos predefinidos) vs **agente** (el LLM dirige su propio flujo).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

# 1) El estado suma un campo: a quién llamar a continuación
class EstadoSupervisor(EstadoEquipo):
    siguiente: str

# 2) El orquestador decide con SALIDA ESTRUCTURADA (más confiable que texto libre)
class Decision(BaseModel):
    siguiente: Literal['investigador', 'redactor', 'revisor', 'FIN'] = Field(
        description="Próximo agente, o 'FIN' si el informe ya está revisado y listo.")
    motivo: str = Field(description='Justificación breve de la decisión.')

SUP_PROMPT = '''Sos el ORQUESTADOR de un equipo. Agentes disponibles:
- investigador: junta información con búsqueda web.
- redactor: escribe el borrador a partir de la investigación.
- revisor: critica el borrador y produce la versión final.

Mirá el estado actual y decidí el PRÓXIMO agente, o 'FIN' si el informe ya fue revisado y está listo.
Orden lógico habitual: investigador -> redactor -> revisor -> FIN. No repitas pasos ya hechos.
Si el revisor pidiera cambios de fondo, podés volver a redactor o investigador.'''

def orquestador_real(state: EstadoSupervisor) -> dict:
    estado = (f"tarea: {state.get('tarea','')}\n"
              f"¿hay investigación?: {'sí' if state.get('investigacion') else 'no'}\n"
              f"¿hay borrador?: {'sí' if state.get('borrador') else 'no'}\n"
              f"¿hay versión final?: {'sí' if state.get('final') else 'no'}")
    d = llm.with_structured_output(Decision).invoke(
        [{'role': 'system', 'content': SUP_PROMPT},
         {'role': 'user', 'content': estado}])
    print(f'🧭 [ORQUESTADOR] decide -> {d.siguiente}   ({d.motivo})')
    return {'siguiente': d.siguiente}

# 3) Traduce la decisión a un destino del grafo
def ruta(state: EstadoSupervisor):
    return END if state['siguiente'] == 'FIN' else state['siguiente']

# 4) Grafo hub-and-spoke: cada worker VUELVE al orquestador (reusamos los agentes de antes)
b = StateGraph(EstadoSupervisor)
b.add_node('orquestador', orquestador_real)
b.add_node('investigador', investigador)
b.add_node('redactor', redactor)
b.add_node('revisor', revisor)
b.add_edge(START, 'orquestador')
b.add_conditional_edges('orquestador', ruta,
    {'investigador': 'investigador', 'redactor': 'redactor', 'revisor': 'revisor', END: END})
for w in ['investigador', 'redactor', 'revisor']:
    b.add_edge(w, 'orquestador')
app_sup = b.compile()

# Como hay loops, ponemos un tope de pasos (recursion_limit) por las dudas.
res = app_sup.invoke(
    {'tarea': 'Escribí un resumen corto sobre qué es un agente LLM y para qué sirve, con un ejemplo.'},
    {'recursion_limit': 15},
)
print('\n=== VERSIÓN FINAL ===\n')
print(res.get('final') or res.get('borrador') or '(sin salida)')

**Reto para los alumnos:** hacé que el revisor pueda **rechazar** el borrador (p. ej. devolviendo un flag en el estado) y que el orquestador, al verlo, **vuelva a `redactor`** en vez de ir a `FIN`. Ahí vas a ver el loop de verdad. También podés agregar un agente nuevo (p. ej. `verificador_de_fuentes`) sin tocar el flujo: solo lo registrás y el orquestador aprende a llamarlo desde el `SUP_PROMPT`.